In [1]:
# Importar librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Cargar el dataset MNIST
print("Cargando dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, parser='auto')
X, y = mnist.data, mnist.target

# Dividir en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train unique values: {np.unique(y_train)}")


Cargando dataset MNIST...


ImportError: Returning pandas objects requires pandas to be installed. Alternatively, explicitly set `as_frame=False` and `parser='liac-arff'`.

## Clasificadores para detección de números pares y números en el intervalo [5, 7]

Primero, convertiremos las etiquetas `y_train` y `y_test` a tipo entero para poder realizar operaciones matemáticas.

In [1]:
y_train_int = y_train.astype(np.int8)
y_test_int = y_test.astype(np.int8)

print(f"y_train_int type: {y_train_int.dtype}")
print(f"y_test_int type: {y_test_int.dtype}")

NameError: name 'y_train' is not defined

### Preparación de las etiquetas para los nuevos clasificadores

Crearemos las etiquetas binarias para las dos nuevas tareas de clasificación:
1.  **Dígito es par**: `True` si el dígito es par, `False` en caso contrario.
2.  **Dígito en [5, 7]**: `True` si el dígito es 5, 6 o 7, `False` en caso contrario.

In [ ]:
# Etiquetas para 'dígito es par'
y_train_even = (y_train_int % 2 == 0)
y_test_even = (y_test_int % 2 == 0)

# Etiquetas para 'dígito en [5, 7]'
y_train_5_7 = ((y_train_int >= 5) & (y_train_int <= 7))
y_test_5_7 = ((y_test_int >= 5) & (y_test_int <= 7))

print(f"Primeros 5 valores de y_train_even: {y_train_even[:5].tolist()}")
print(f"Primeros 5 valores de y_train_5_7: {y_train_5_7[:5].tolist()}")

### Entrenamiento de los cuatro clasificadores

Entrenaremos un `SGDClassifier` y un `KNeighborsClassifier` para cada una de las dos tareas binarias. Utilizaremos el conjunto de entrenamiento (`X_train`).

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier

# 1. SGDClassifier para 'dígito es par'
sgd_clf_even = SGDClassifier(random_state=42)
sgd_clf_even.fit(X_train, y_train_even)
print("SGDClassifier (even) entrenado.")

# 2. KNeighborsClassifier para 'dígito es par'
knn_clf_even = KNeighborsClassifier()
knn_clf_even.fit(X_train, y_train_even)
print("KNeighborsClassifier (even) entrenado.")

# 3. SGDClassifier para 'dígito en [5, 7]'
sgd_clf_5_7 = SGDClassifier(random_state=42)
sgd_clf_5_7.fit(X_train, y_train_5_7)
print("SGDClassifier (5-7) entrenado.")

# 4. KNeighborsClassifier para 'dígito en [5, 7]'
knn_clf_5_7 = KNeighborsClassifier()
knn_clf_5_7.fit(X_train, y_train_5_7)
print("KNeighborsClassifier (5-7) entrenado.")

### Evaluación de los clasificadores en el conjunto de prueba

Ahora evaluaremos el rendimiento de cada clasificador en el conjunto de prueba (`X_test`) utilizando las métricas de precisión, recall, F1-score y AUC.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve

def evaluate_classifier(clf, X_test_data, y_test_true, name):
    print(f"--- Evaluación de {name} ---")
    y_pred = clf.predict(X_test_data)

    # Calcular scores de decisión o probabilidades para AUC y ROC
    if hasattr(clf, "decision_function"):
        y_scores = clf.decision_function(X_test_data)
    elif hasattr(clf, "predict_proba"):
        y_scores = clf.predict_proba(X_test_data)[:, 1]
    else:
        y_scores = y_pred # Fallback, but might not be suitable for ROC/AUC

    precision = precision_score(y_test_true, y_pred)
    recall = recall_score(y_test_true, y_pred)
    f1 = f1_score(y_test_true, y_pred)
    auc = roc_auc_score(y_test_true, y_scores)

    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-score: {f1:.4f}")
    print(f"  AUC: {auc:.4f}")
    return y_scores

# Evaluación SGDClassifier (even)
y_scores_sgd_even = evaluate_classifier(sgd_clf_even, X_test, y_test_even, "SGDClassifier (Even)")

# Evaluación KNeighborsClassifier (even)
y_scores_knn_even = evaluate_classifier(knn_clf_even, X_test, y_test_even, "KNeighborsClassifier (Even)")

# Evaluación SGDClassifier (5-7)
y_scores_sgd_5_7 = evaluate_classifier(sgd_clf_5_7, X_test, y_test_5_7, "SGDClassifier (5-7)")

# Evaluación KNeighborsClassifier (5-7)
y_scores_knn_5_7 = evaluate_classifier(knn_clf_5_7, X_test, y_test_5_7, "KNeighborsClassifier (5-7)")

### Curva ROC de los cuatro clasificadores

Finalmente, graficaremos las curvas ROC para todos los clasificadores en un mismo gráfico para comparar visualmente su rendimiento.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

# Curva ROC para SGDClassifier (even)
fpr_sgd_even, tpr_sgd_even, _ = roc_curve(y_test_even, y_scores_sgd_even)
plt.plot(fpr_sgd_even, tpr_sgd_even, "b-", linewidth=2, label="SGDClassifier (Even)")

# Curva ROC para KNeighborsClassifier (even)
fpr_knn_even, tpr_knn_even, _ = roc_curve(y_test_even, y_scores_knn_even)
plt.plot(fpr_knn_even, tpr_knn_even, "g-", linewidth=2, label="KNeighborsClassifier (Even)")

# Curva ROC para SGDClassifier (5-7)
fpr_sgd_5_7, tpr_sgd_5_7, _ = roc_curve(y_test_5_7, y_scores_sgd_5_7)
plt.plot(fpr_sgd_5_7, tpr_sgd_5_7, "r-", linewidth=2, label="SGDClassifier (5-7)")

# Curva ROC para KNeighborsClassifier (5-7)
fpr_knn_5_7, tpr_knn_5_7, _ = roc_curve(y_test_5_7, y_scores_knn_5_7)
plt.plot(fpr_knn_5_7, tpr_knn_5_7, "m-", linewidth=2, label="KNeighborsClassifier (5-7)")

# Curva ROC de clasificador aleatorio
plt.plot([0, 1], [0, 1], 'k:', label="Clasificador aleatorio")

plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR) / Recall')
plt.title('Curvas ROC de los Cuatro Clasificadores')
plt.grid(True)
plt.legend(loc="lower right", fontsize=13)
plt.show()